# Project 2: The Two-Dimensional Ising Model

The 2D Ising model on a square lattice of $N = L \times L$ sites is defined by the Hamiltonian:

$$\mathcal{H} = -J \sum_{\langle i,j \rangle} \sigma_i \sigma_j$$

where $\sigma_i = \pm 1$ represents an Ising spin at site $i$, and $J > 0$ is the ferromagnetic coupling between nearest neighbors $\langle i,j \rangle$. We will use periodic boundary conditions.

This simple model is the "fruit fly" of statistical physics. At low temperature, energy wins and the spins align; at high temperature, entropy wins and they point randomly. In 1944 Onsager solved the model exactly and showed that on the infinite lattice ($N \rightarrow \infty$) the competition is settled by a genuine **phase transition** at a temperature $T_c$ between a disordered state at high temperature and a magnetic state with a finite magnetization $\langle m \rangle \ne 0$, where

\begin{equation}
  \langle m \rangle = \lim_{N \rightarrow \infty}
     \frac{1}{N} \sum_{i=1}^N \langle \sigma_i \rangle
\end{equation}

The value of the critical temperature (we set $k_B=1$) is

\begin{equation}
  \frac{T_c}{J} = \frac{2}{\log (1 + \sqrt{2})} \simeq 2.269
\end{equation}

and the magnetization and energy are known exactly too:

\begin{equation}
  \langle m \rangle = \begin{cases} \big(1 - \sinh^{-4}(2J/T)\big)^{1/8}, & T < T_c \\ 0, & T > T_c \end{cases}
\end{equation}

\begin{equation}
  \langle e \rangle = -J \coth(2J/T) \left[ 1 + \frac{2}{\pi} \big(2\tanh^2(2J/T) - 1\big) \mathrm{K}(k) \right]
\end{equation}

where $k = \frac{2\sinh(2K)}{\cosh^2(2K)}$ and $\mathrm{K}(k)$ is the complete elliptic integral of the first kind.

The goal of this project is to study the two-dimensional Ising model with a Monte Carlo algorithm, and to see whether we can find evidence for the phase transition even though we will only ever be dealing with finite systems (where, strictly speaking, there is no phase transition at all!).

In [ ]:
# Common imports
%matplotlib inline
from collections.abc import Mapping

import numpy as np
from scipy.special import ellipk

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap
from matplotlib.image import AxesImage
from IPython.display import HTML

# the random number generator
rng = np.random.default_rng(seed=2026)

PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9", "#E69F00", "#F0E442"]
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (6.5, 4.2),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.edgecolor": "#898781",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.axisbelow": True,
    "animation.html": "jshtml",    # animations are rendered as interactive JavaScript widgets
    "animation.embed_limit": 60,   # in MB
})

# colors for spin down / spin up, and for highlighting clusters
SPIN_CMAP = ListedColormap(["#f0efec", PALETTE[0]])
def show_spins(spins: np.ndarray, ax: plt.Axes | None = None) -> AxesImage:
    """Draw a spin configuration (down = light, up = blue) and return the image artist."""
    ax = ax if ax is not None else plt.gca()
    im = ax.imshow(spins, cmap=SPIN_CMAP, vmin=-1, vmax=1, interpolation="nearest")
    ax.set(xticks=[], yticks=[])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
    return im

# the critical temperature
T_C = 2 / np.log(1 + np.sqrt(2))

# the formula for the energy
def onsager_energy(T: np.ndarray | float, J: float = 1.0) -> np.ndarray:
    """Energy per spin of the infinite lattice."""
    K = J / np.asarray(T, dtype=float)
    k = 2 * np.sinh(2 * K) / np.cosh(2 * K) ** 2
    return -J / np.tanh(2 * K) * (1 + 2 / np.pi * (2 * np.tanh(2 * K) ** 2 - 1) * ellipk(k**2))

# the formula for the magnetization
def onsager_magnetization(T: np.ndarray | float, J: float = 1.0) -> np.ndarray:
    """Spontaneous magnetization per spin of the infinite lattice (Yang)."""
    T = np.asarray(T, dtype=float)
    below = T < 2 * J / np.log(1 + np.sqrt(2))
    x = np.sinh(2 * J / np.where(below, T, 1.0)) ** -4
    return np.where(below, np.clip(1 - x, 0, None) ** 0.125, 0.0)

## Warmup: Compute $\pi$ by Monte Carlo

In order to get familiar with the generation of random numbers, you can try to write a
Monte Carlo algorithm to compute $\pi$. The idea, as we saw in the lecture, is to start
from a circle of radius 1, inside a square box of length 2. You then throw randomly
(uniformly) stones in the square box. After having thrown many stones you can compute
the fraction of those that landed inside the circle. This will give you an estimate
for $\pi$.

Random numbers in modern numpy are drawn from a *generator* object:

```python
rng = np.random.default_rng(seed=42)   # the seed makes the run reproducible
rng.uniform(-1, 1, size=(10, 2))       # 10 random points in the square [-1, 1]^2
```

*Challenge*: Show that the error bar on the value of $\pi$ goes like $1 / \sqrt{M}$, where
$M$ is the number of stones. To do this, you will need to make several runs (about 100) for
the same fixed $M$ and compute the standard deviation of the estimates.

## Solution

We throw $M$ stones uniformly in the square $[-1,1]^2$. The probability for a stone to land inside
the unit disc is $p = \pi/4$, so $4 \times (\text{fraction inside})$ is an estimate of $\pi$.
The number of stones inside follows a binomial distribution and the standard deviation of the estimate is

\begin{equation}
  \sigma_\pi = 4 \sqrt{\frac{p(1-p)}{M}} = \frac{\sqrt{\pi(4-\pi)}}{\sqrt{M}}
\end{equation}

To measure it, we repeat the simulation `n_runs` times for each $M$ and compute the standard deviation
of the estimates.

In [ ]:
def estimate_pi(n_stones: int, rng: np.random.Generator) -> float:
    """One Monte Carlo estimate of pi using `n_stones` stones."""
    x, y = rng.uniform(-1, 1, size=(2, n_stones))
    samples = 4 * ((x**2 + y**2) < 1)
    return np.average(samples)


n_runs = 200
n_stones = np.logspace(1, 6, 11).astype(int)
error_bar = np.array([np.std([estimate_pi(M, rng) for _ in range(n_runs)]) for M in n_stones])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), width_ratios=[1, 1.35], layout="constrained")

# left: what a single run looks like
x, y = rng.uniform(-1, 1, size=(2, 3000))
inside = x**2 + y**2 < 1
ax1.scatter(x[inside], y[inside], s=3, color=PALETTE[0])
ax1.scatter(x[~inside], y[~inside], s=3, color=PALETTE[1])
ax1.add_patch(plt.Circle((0, 0), 1, fill=False, color="black", lw=2.0))
ax1.set(aspect="equal", xlim=(-1.02, 1.02), ylim=(-1.02, 1.02), xticks=[-1, 0, 1], yticks=[-1, 0, 1],
        title=fr"$M = 3000$:  $\pi \approx {4 * inside.mean():.3f}$")
ax1.grid(False)

# right: the error bar decreases like 1 / sqrt(M)
ax2.loglog(n_stones, error_bar, "o", ms=7, label=f"std over {n_runs} runs")
ax2.loglog(n_stones, np.sqrt(np.pi * (4 - np.pi) / n_stones), "--", label=r"$\sqrt{\pi(4-\pi)/M}$")
ax2.set(xlabel="number of stones $M$", ylabel=r"$\sigma_\pi$", title=r"Error bar on $\pi$")
ax2.legend();

## Part 1: Set up a class for the Ising model

Let's first start and write a class that will describe the Ising model. The class
should have a constructor taking $T$, $J$, $L$ and a random number generator as parameters. It generates
a random initial state which can be encoded in an $L\times L$
`numpy.ndarray` of $\pm 1$. We use periodic boundary conditions. The class could have a structure like

```python
class Configuration:
    def __init__(self, T: float, J: float, L: int, rng: np.random.Generator) -> None:
        self.size = L
        self.spins = ...
        ...

    def get_energy(self) -> float:
        ...

    def get_magnetization(self) -> int:
        ...
```

In order to test your class, you can instantiate it and plot the spin configuration
using the `show_spins` function defined above. You can also check your class methods by generating
special configurations with all spins up or a checkerboard (Néel) pattern and see whether the energy
and magnetization are those you expect.

*Hint*: `np.roll(spins, -1, axis=1)` shifts the array by one site, with periodic
boundary conditions. It gives you all the right neighbors at once, without any loop.

## Solution

Here is the class for the configuration. It creates a random initial state and sets two member
variables: the energy and the magnetization. The energy is computed without any explicit loop:
`np.roll(spins, -1, axis=1)` is the array of right neighbors and `np.roll(spins, -1, axis=0)` the array
of bottom neighbors (with periodic boundary conditions), so that each bond is counted exactly once.

The configuration also carries its own random number generator `rng`, which will be used by the
Monte Carlo moves. Passing a generator around (rather than using a global random state) is the
modern way, and it makes simulations reproducible.

In [ ]:
class Configuration:
    """A configuration of Ising spins on an L x L square lattice with periodic boundary conditions."""

    def __init__(self, T: float, J: float, L: int, rng: np.random.Generator) -> None:
        self.size = L
        self.J = J
        self.T = T
        self.beta = 1.0 / T
        self.rng = rng
        self.spins = self.rng.choice([-1, 1], size=(L, L))  # either +1 or -1
        self.energy = self.get_energy()
        self.magnetization = self.get_magnetization()

    def get_energy(self) -> float:
        """Total energy, computed from scratch."""
        s = self.spins
        return float(-self.J * np.sum(s * (np.roll(s, -1, axis=0) + np.roll(s, -1, axis=1))))

    def get_magnetization(self) -> int:
        """Total magnetization, computed from scratch."""
        return int(np.sum(self.spins))

Let's check a random state (which should look random!) and two simple configurations whose energy we know:
the fully polarized state has $E = -2JN$ and the checkerboard (Néel) state has $E = +2JN$.

In [ ]:
L = 50
N = L**2

# random
config = Configuration(T=4.0, J=1.0, L=L, rng=rng)

# all spins in the same direction
polarized = Configuration(T=4.0, J=1.0, L=L, rng=rng)
polarized.spins = np.ones((L, L), dtype=int)

# a checkerboard pattern
neel = Configuration(T=4.0, J=1.0, L=L, rng=rng)
i, j = np.indices((L,L))
neel.spins = np.where((i + j) % 2 == 0, 1, -1)

assert polarized.get_energy() == -2 * N and polarized.get_magnetization() == N
assert neel.get_energy() == 2 * N and neel.get_magnetization() == 0

fig, axes = plt.subplots(1, 3, figsize=(10, 3.8), layout="constrained")
for ax, (name, c) in zip(axes, [("random", config), ("polarized", polarized), ("Néel", neel)]):
    show_spins(c.spins, ax)
    ax.set_title(f"{name}\n$E/N = {c.get_energy() / N:+.2f}$,  $m = {c.get_magnetization() / N:+.2f}$",
                 fontsize=12)

## Part 2: The Metropolis move

Here you will write a function that performs a Metropolis Monte Carlo
move on the configuration. The function takes an Ising configuration
(an instance of the `Configuration` class) as an argument and has
the following signature:

```python
def metropolis_move(config: Configuration) -> None:
    ...
```

The function should choose a random site, compute the energy difference
between the old configuration and the new configuration where the spin
has been flipped and decide whether the move should be accepted or
not (using the Metropolis algorithm). Warning: in order to compute the
energy difference, only consider the energy change from the links involving
the flipped spin. Indeed, all the other links are unchanged and it would
be a waste of time to recompute the full energy every time.

How to check? Start from a small lattice, say $4 \times 4$, and call
`metropolis_move` many times at a rather high temperature. This way,
you should sample many different configurations. If your energy calculation
is right, the energy per spin should always be between $-2J$ and $2J$, and the energy
you update after each move should always agree with the one recomputed from scratch.

## Solution

Flipping the spin $\sigma_i$ only changes the energy of the four bonds attached to it:

\begin{equation}
  \Delta E = 2 J \sigma_i \sum_{j \in \text{nn}(i)} \sigma_j \in \{-8J, -4J, 0, 4J, 8J\}
\end{equation}

The move is accepted with probability $\min(1, e^{-\beta \Delta E})$. Moves that lower the energy
($\Delta E \le 0$) are always accepted, so we only need to draw a random number otherwise.
Note that `s[i - 1, j]` with `i = 0` is `s[-1, j]`, the last row: negative indices give the periodic
boundary conditions for free, we only need a modulo for `i + 1`.

In [ ]:
def metropolis_move(config: Configuration) -> None:
    """Try to flip one random spin with the Metropolis acceptance rule."""
    L, s = config.size, config.spins
    i, j = config.rng.integers(L, size=2)  # pick a random site

    # energy difference, from the four bonds attached to (i, j) only
    delta_energy = 2 * config.J * s[i, j] * (s[(i + 1)%L, j] + s[i - 1, j] + s[i, (j + 1)%L] + s[i, j - 1])

    # accept with the Metropolis probability, otherwise leave the configuration unchanged
    if delta_energy <=0 or config.rng.random() < np.exp(-config.beta * delta_energy):
        s[i, j] *= -1
        config.energy += delta_energy
        config.magnetization += 2 * s[i, j]

This is correct, but each call to the random number generator has a sizable overhead in Python. It is
much faster to draw all the random sites and random numbers needed for a *sweep* ($N = L^2$ attempted
flips) at once. Since $\Delta E$ can only take a few values, we also precompute the acceptance
probabilities $e^{-\beta \Delta E}$ once and for all. The algorithm is exactly the same, but
`metropolis_sweep` is about ten times faster than calling `metropolis_move` $N$ times. We use it
from now on.

In [ ]:
def metropolis_sweep(config: Configuration, n_moves: int | None = None) -> None:
    """Perform `n_moves` Metropolis moves (one sweep, L*L moves, by default)."""
    L, J, s = config.size, config.J, config.spins
    n_moves = L * L if n_moves is None else n_moves
    acceptance = {dE: np.exp(-config.beta * dE) for dE in (4 * J, 8 * J)}

    sites = config.rng.integers(L, size=(n_moves, 2)).tolist()
    uniforms = config.rng.random(n_moves).tolist()
    for (i, j), u in zip(sites, uniforms):
        spin = int(s[i, j])
        delta_energy = 2 * J * spin * int(s[(i + 1)%L, j] + s[i - 1, j] + s[i, (j + 1)%L] + s[i, j - 1])
        if delta_energy <= 0 or u < acceptance[delta_energy]:
            s[i, j] = -spin
            config.energy += delta_energy
            config.magnetization -= 2 * spin

Let's check both functions on a $4 \times 4$ lattice: the running energy must stay in $[-2JN, 2JN]$ and agree
with the energy recomputed from scratch. We also look at how the energy is distributed at a high and a low
temperature.

In [ ]:
import time

n_moves = 50_000
fig, ax = plt.subplots()
for T, color in [(10.0, PALETTE[1]), (2.0, PALETTE[0])]:
    config = Configuration(T=T, J=1.0, L=4, rng=rng)
    energies = np.empty(n_moves)
    for k in range(n_moves):
        metropolis_move(config)
        energies[k] = config.energy / 16
    assert np.all(np.abs(energies) <= 2)
    assert config.energy == config.get_energy() and config.magnetization == config.get_magnetization()
    values, counts = np.unique(energies, return_counts=True)
    ax.bar(values, counts / n_moves, width=0.2, alpha=0.8, color=color, label=f"$T = {T:g}$")
ax.set(xlabel="energy per spin $E/N$", ylabel="probability", title=r"Energy histogram, $4 \times 4$ lattice")
ax.legend()

# timing comparison on a larger lattice
config = Configuration(T=2.3, J=1.0, L=64, rng=rng)
start = time.perf_counter()
for _ in range(config.size**2):
    metropolis_move(config)
t_move = time.perf_counter() - start
start = time.perf_counter()
metropolis_sweep(config)
t_sweep = time.perf_counter() - start
assert config.energy == config.get_energy() and config.magnetization == config.get_magnetization()
print(f"one sweep of a 64x64 lattice: {1e3 * t_move:.1f} ms with metropolis_move, {1e3 * t_sweep:.1f} ms with metropolis_sweep")

## Part 3: The Monte Carlo simulation live!

Write an animation that shows the evolution of the spins. In order for the animation not to be
too slow, the function that is repeatedly called by `FuncAnimation` should make many spin
flip trials (typically $L\times L$, which we call a *sweep*). Otherwise the animation will be
really slow. You might want to get more details about
[`matplotlib.animation.FuncAnimation`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html)
on the matplotlib website. In a notebook, the simplest way to display the animation is

```python
from IPython.display import HTML

ani = animation.FuncAnimation(fig, update, frames=200, interval=40)
plt.close(fig)            # do not show the static figure as well
HTML(ani.to_jshtml())     # an interactive player with play / pause / speed controls
```

and you can save it to a file with `ani.save("ising.mp4")` or `ani.save("ising.gif")`.

See how the system behaves for different:

- Temperatures
- System sizes

## Solution

Each frame of the animation performs one sweep. On the left we show the spins, and on the right the
magnetization and energy per spin as a function of Monte Carlo time. The dashed lines are Onsager's exact
results for the infinite system (see the section on Onsager's solution above): the spontaneous
magnetization $\pm m_\infty$ and the energy $e_\infty$. We use an `init_func` so that the first frame is drawn without doing a
Monte Carlo step, and `blit=True` so that only the parts of the figure that change are redrawn.

In [ ]:
def animate_metropolis(L: int, T: float, n_frames: int = 200, sweeps_per_frame: int = 1) -> animation.FuncAnimation:
    """Animate a Metropolis simulation starting from a random configuration."""
    config = Configuration(T=T, J=1.0, L=L, rng=rng)
    N = L * L
    times, mags, energies = [0], [config.magnetization / N], [config.energy / N]

    fig = plt.figure(figsize=(10.5, 4.6), layout="constrained", dpi=90)
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.3])
    ax_spins = fig.add_subplot(grid[:, 0])
    ax_m = fig.add_subplot(grid[0, 1])
    ax_e = fig.add_subplot(grid[1, 1], sharex=ax_m)

    image = show_spins(config.spins, ax_spins)
    ax_spins.set_title(fr"$L = {L}$,  $T = {T:g}$  ($T/T_c = {T / T_C:.2f}$)")
    (line_m,) = ax_m.plot(times, mags, color=PALETTE[0])
    (line_e,) = ax_e.plot(times, energies, color=PALETTE[1])
    m_exact = float(onsager_magnetization(T))
    for sign in (1, -1):
        ax_m.axhline(sign * m_exact, color="#898781", ls="--", lw=1.2)
    ax_e.axhline(float(onsager_energy(T)), color="#898781", ls="--", lw=1.2, label="Onsager, $L = \\infty$")
    ax_m.set(xlim=(0, n_frames * sweeps_per_frame), ylim=(-1.05, 1.05), ylabel="magnetization $m$")
    ax_e.set(ylim=(-2.05, 0.05), xlabel="Monte Carlo sweeps", ylabel="energy $E/N$")
    ax_e.legend(loc="upper right", fontsize=10)
    ax_m.tick_params(labelbottom=False)

    def init():
        return image, line_m, line_e

    def update(frame: int):
        for _ in range(sweeps_per_frame):
            metropolis_sweep(config)
        times.append((frame + 1) * sweeps_per_frame)
        mags.append(config.magnetization / N)
        energies.append(config.energy / N)
        image.set_data(config.spins)
        line_m.set_data(times, mags)
        line_e.set_data(times, energies)
        return image, line_m, line_e

    ani = animation.FuncAnimation(fig, update, frames=n_frames, init_func=init, interval=40, blit=True)
    plt.close(fig)
    return ani


# rendering the animation takes about 30 seconds
ani = animate_metropolis(L=64, T=1.3*T_C, n_frames=200, sweeps_per_frame=2)
HTML(ani.to_jshtml())

Below $T_c$, starting from a random configuration, domains of up and down spins form and slowly coarsen:
the system is performing a *quench*. The magnetization will eventually approach one of the two
dashed lines, but this can take a very long time, and the system can even get stuck in a striped
configuration with two domain walls wrapping around the periodic lattice. Try other parameters, e.g.
`animate_metropolis(L=64, T=T_C)` to see fluctuations on all length scales, or
`animate_metropolis(L=64, T=3.5)` for the paramagnetic phase. Use `ani.save("ising.mp4", fps=25)` to
export an animation.

## Part 4: Compute physical averages

The animation above allows to get some insight into the physics of the
model. But it is necessary to compute some physical averages to understand
more. Because two configurations that only differ by a spin flip are very
correlated, one usually waits a certain number of steps before considering
a new configuration in the computation of an average. In practice, the
Monte Carlo simulation can be cut into cycles: a cycle is a certain number
of steps needed to decorrelate the configurations. Also, in the beginning of
the simulation, one needs to wait for the Markov chain to reach its stationary
distribution. These variables can be used:

- `length_cycle`: the number of steps between two *measurements*, i.e. between two
   configurations that are used to compute physical averages.

- `n_cycles`: the number of cycles used to compute the averages. This number
   corresponds to the number of measurements.

- `n_warmup`: the number of cycles that are performed in the beginning of the
   simulation without any measurement. They are done so that the Markov
   chain reaches a stationary distribution.

You can now write a Monte Carlo simulation:

1. Compute the average magnetization $m$ as a function of the number of
  Monte Carlo steps on a $20 \times 20$ lattice. This will allow you to
  judge how many steps are necessary to reach equilibrium. How
  does this change with temperature and system size?

2. You can then more systematically compute the magnetization, the energy, the
  magnetic susceptibility and specific heat as a function of the temperature.
  Start with a rather small lattice $4 \times 4$ and then increase the size.
  How do these quantities vary? Recall that

  \begin{equation}
    c = \frac{\langle E^2 \rangle - \langle E \rangle^2}{N T^2}
    \qquad
    \chi = \frac{\langle M^2 \rangle - \langle M \rangle^2}{N T}
  \end{equation}

  where $E$ and $M$ are the total energy and magnetization.

You will see that these simulations can take some time. Rather than
plotting directly after the simulation is done, you might want to save
your averages into files and in another cell read the information from these
files to do the plots. It is very common to organize a numerical work like
this: a production part that generates the data and a postprocessing part
to do the plots etc. To save and read numpy arrays you can use:

```python
np.savez("results_L4.npz", temperature=temperatures, energy=energy)
data = np.load("results_L4.npz")
data["energy"]
```

## Solution

### Equilibration

We first plot the magnetization as a function of Monte Carlo time for a few temperatures,
starting from a random configuration. The dashed lines are Onsager's $\pm m_\infty(T)$ for $T < T_c$.

In [ ]:
L, n_sweeps = 20, 1000
fig, ax = plt.subplots(figsize=(8, 4.2))
for T in [1.5, 2.0, 2.3, 3.0]:
    config = Configuration(T=T, J=1.0, L=L, rng=rng)
    mags = np.empty(n_sweeps)
    for t in range(n_sweeps):
        metropolis_sweep(config)
        mags[t] = config.magnetization / L**2
    (line,) = ax.plot(mags, lw=1.4, label=f"$T = {T:g}$")
    if T < T_C:  # Onsager's spontaneous magnetization
        for sign in (1, -1):
            ax.axhline(sign * onsager_magnetization(T), color=line.get_color(), ls="--", lw=1)
ax.set(xlabel="Monte Carlo sweeps", ylabel="magnetization $m$", ylim=(-1.05, 1.05),
       title=fr"Equilibration of a ${L} \times {L}$ lattice")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5));

At high temperature the magnetization fluctuates around zero after a handful of sweeps. At low temperature
it reaches $\pm m_\infty$ after typically a few hundred sweeps (it depends on the random start, and grows
with the system size). Close to $T_c$ the fluctuations are large and slow: $m$ wanders between positive and
negative values. A warmup of $\sim 1000$ sweeps is a safe choice for these small lattices.

### Physical averages

A finite system never truly breaks the $\sigma \to -\sigma$ symmetry: given enough time, it flips between
$+m$ and $-m$, so that $\langle M \rangle = 0$ at all temperatures. This is very visible on a $4 \times 4$
lattice. The standard remedy is to measure $\langle |M| \rangle$ instead, and to define the susceptibility
as $\chi = (\langle M^2 \rangle - \langle |M| \rangle^2)/(NT)$.

The function below runs a simulation at a given temperature and returns the averages. This is the
production part, and the results are saved in a file.

In [ ]:
def simulate(T: float, L: int, n_cycles: int, n_warmup: int, length_cycle: int | None = None) -> dict[str, float]:
    """Metropolis simulation at temperature T on an L x L lattice. Returns the physical averages per spin."""
    config = Configuration(T=T, J=1.0, L=L, rng=rng)
    length_cycle = L * L if length_cycle is None else length_cycle
    for _ in range(n_warmup):
        metropolis_sweep(config, length_cycle)

    E, M = np.empty(n_cycles), np.empty(n_cycles)
    for n in range(n_cycles):
        metropolis_sweep(config, length_cycle)
        E[n], M[n] = config.energy, abs(config.magnetization)

    N = L * L
    return {
        "energy": E.mean() / N,
        "magnetization": M.mean() / N,
        "specific_heat": E.var() / (N * T**2),
        "susceptibility": M.var() / (N * T)
    }

In [ ]:
# production: this takes about 10 seconds
from tqdm.auto import tqdm

L = 4
temperatures = np.concatenate([np.linspace(0.5, 1.75, 6), np.linspace(2.0, 2.6, 13), np.linspace(2.9, 5.0, 8)])
results = [simulate(T, L, n_cycles=10_000, n_warmup=1000) for T in tqdm(temperatures, desc=f"L = {L}")]

np.savez(f"ising_L{L}.npz", temperature=temperatures, **{key: [r[key] for r in results] for key in results[0]})

Here comes the analysis of the data. We compare with Onsager's exact results for the infinite lattice
(dashed lines).

In [ ]:
def plot_observables(datasets: dict[str, Mapping[str, np.ndarray]], axes=None, **plot_kwargs):
    """Plot energy, |m|, specific heat and susceptibility for several datasets, with the exact L = infinity result."""
    if axes is None:
        fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, layout="constrained")
    ax_e, ax_m, ax_c, ax_chi = axes.flat

    T_all = np.concatenate([data["temperature"] for data in datasets.values()])
    T_fine = np.linspace(T_all.min(), T_all.max(), 2000)
    exact = dict(color="#0b0b0b", ls="--", lw=1.4, label=r"Onsager, $L = \infty$")
    ax_e.plot(T_fine, onsager_energy(T_fine), **exact)
    ax_m.plot(T_fine, onsager_magnetization(T_fine), **exact)

    for label, data in datasets.items():
        T = data["temperature"]
        kwargs = dict(marker="o", ms=4.5, label=label) | plot_kwargs
        ax_e.plot(T, data["energy"], **kwargs)
        ax_m.plot(T, data["magnetization"], **kwargs)
        ax_c.plot(T, data["specific_heat"], **kwargs)
        ax_chi.plot(T, data["susceptibility"], **kwargs)

    for ax, ylabel in zip(axes.flat, ["energy $e = E/N$", r"magnetization $\langle |m| \rangle$",
                                      "specific heat $c$", r"susceptibility $\chi$"]):
        ax.axvline(T_C, color="#898781", lw=1)
        ax.set_ylabel(ylabel)
    
    ax_c.set(xlabel="temperature $T/J$", ylim=(0, 2.00))
    ax_chi.set(xlabel="temperature $T/J$", yscale="log")
    ax_e.text(T_C + 0.05, -1.9, "$T_c$", color="#52514e")
    chi_max = max(np.nanmax(data["susceptibility"]) for data in datasets.values())
    ax_chi.set_ylim(1e-3, 3 * chi_max)
    ax_m.legend(loc="upper right", fontsize=10)
    return axes


data = np.load("ising_L4.npz")
axes = plot_observables({r"$4 \times 4$": data})
axes[0, 0].figure.suptitle(r"Metropolis simulation of a $4 \times 4$ lattice", fontsize=15);

Even on this tiny lattice the qualitative features of the transition are visible: the magnetization
decreases rapidly around $T_c$, and the specific heat and the susceptibility have a maximum near $T_c$.
But the finite-size effects are large: the magnetization remains finite above $T_c$ and the maxima are
broad and shifted. To see the transition emerge, we need larger lattices.

### Going further: a vectorized (checkerboard) algorithm

With a Python loop over the spins, larger lattices quickly become slow. We can do much better with numpy
by noticing that the square lattice is *bipartite*: color the sites like a checkerboard, then the
neighbors of a black site are all white. The energy change of flipping a black spin therefore only
depends on white spins, and **all black spins can be updated at the same time**, independently. We then
update all white spins. One such double step is a sweep in which every spin is visited once, in a fixed
order rather than a random one. Each half-step satisfies detailed balance, so the sweep samples the same
Boltzmann distribution.

The function below does one checkerboard Metropolis sweep of a `Configuration`, and keeps its energy and
magnetization up to date. Then `simulate_checkerboard` is the same as `simulate` above, with
`metropolis_sweep` replaced by `checkerboard_sweep`.

In [ ]:
def checkerboard_sweep(config: Configuration) -> None:
    """One checkerboard Metropolis sweep: all black sites, then all white sites."""
    L, s = config.size, config.spins
    assert L % 2 == 0, "the checkerboard update needs an even L"
    i, j = np.indices((L, L))
    for color in (0, 1):
        sublattice = (i + j) % 2 == color
        neighbors = np.roll(s, 1, axis=0) + np.roll(s, -1, axis=0) + np.roll(s, 1, axis=1) + np.roll(s, -1, axis=1)
        delta_energy = 2 * config.J * s * neighbors
        accepted = sublattice & (config.rng.random((L, L)) < np.exp(-config.beta * delta_energy))
        s[accepted] *= -1
        config.energy += delta_energy[accepted].sum()
        config.magnetization += 2 * s[accepted].sum()


def simulate_checkerboard(T: float, L: int, n_cycles: int, n_warmup: int) -> dict[str, float]:
    """Checkerboard Metropolis simulation at temperature T on an L x L lattice, starting from all spins up."""
    config = Configuration(T=T, J=1.0, L=L, rng=rng)
    config.spins = np.ones((L, L), dtype=int)  # cold start, see below
    config.energy, config.magnetization = config.get_energy(), config.get_magnetization()
    for _ in range(n_warmup):
        checkerboard_sweep(config)

    E, M = np.empty(n_cycles), np.empty(n_cycles)
    for n in range(n_cycles):
        checkerboard_sweep(config)
        E[n], M[n] = config.energy, abs(config.magnetization)

    N = L * L
    return {
        "energy": E.mean() / N,
        "magnetization": M.mean() / N,
        "specific_heat": E.var() / (N * T**2),
        "susceptibility": M.var() / (N * T),  # M holds |M|, so this is <M^2> - <|M|>^2
        "binder": 1 - np.mean(M**4) / (3 * np.mean(M**2) ** 2),
    }

We can now afford to animate large lattices. Here are three $128 \times 128$ lattices, started from the
same random configuration and quenched to three temperatures, below, at and above $T_c$:

In [ ]:
def animate_quench(L: int = 128, temperatures=(1.5, T_C, 3.5), n_frames: int = 150,
                   sweeps_per_frame: int = 2) -> animation.FuncAnimation:
    """Animate lattices started from the same random configuration, at different temperatures."""
    configs = [Configuration(T=T, J=1.0, L=L, rng=rng) for T in temperatures]
    for config in configs[1:]:  # same initial state for all temperatures
        config.spins = configs[0].spins.copy()
        config.energy, config.magnetization = config.get_energy(), config.get_magnetization()
    N = L * L
    times = [0]
    mags = [[config.magnetization / N] for config in configs]

    fig = plt.figure(figsize=(11, 6.2), layout="constrained", dpi=80)
    grid = fig.add_gridspec(2, len(configs), height_ratios=[2.1, 1])
    ax_m = fig.add_subplot(grid[1, :])
    images, lines = [], []
    for k, config in enumerate(configs):
        color = PALETTE[k + 1]
        ax = fig.add_subplot(grid[0, k])
        images.append(show_spins(config.spins, ax))
        ax.set_title(fr"$T/T_c = {config.T / T_C:.2f}$", color=color)
        lines.append(ax_m.plot(times, mags[k], color=color)[0])
        if config.T < T_C:  # Onsager's spontaneous magnetization
            for sign in (1, -1):
                ax_m.axhline(sign * onsager_magnetization(config.T), color=color, ls="--", lw=1.2)
    ax_m.set(xlim=(0, n_frames * sweeps_per_frame), ylim=(-1.05, 1.05),
             xlabel="Monte Carlo sweeps", ylabel="magnetization $m$")

    def init():
        return *images, *lines

    def update(frame: int):
        times.append((frame + 1) * sweeps_per_frame)
        for config, image, line, m in zip(configs, images, lines, mags):
            for _ in range(sweeps_per_frame):
                checkerboard_sweep(config)
            m.append(config.magnetization / N)
            image.set_data(config.spins)
            line.set_data(times, m)
        return *images, *lines

    ani = animation.FuncAnimation(fig, update, frames=n_frames, init_func=init, interval=50, blit=True)
    plt.close(fig)
    return ani


# rendering the animation takes about 30 seconds
HTML(animate_quench().to_jshtml())

Below $T_c$ (left) domains coarsen: their typical size grows like $\sqrt{t}$, and the magnetization only
slowly approaches Onsager's value (dashed lines). Above $T_c$ (right) the system
quickly equilibrates to a disordered state with small clusters. At $T_c$ (middle) there are clusters of all
sizes: this is the signature of a diverging correlation length.

Let's now compute the physical averages for several lattice sizes, looping over the temperatures as in 4.2.
Note that we start from the fully polarized state (a *cold start*) rather than from a random one: as
the animation shows, a random start at low temperature can leave the lattice in a striped state, with two
domain walls wrapping around it, that survives for a very long time. At high temperature, the polarized
state melts in a few sweeps.

In [ ]:
# production: this takes about 1.5 minutes
temperatures = np.concatenate([np.linspace(1.0, 1.75, 4), np.linspace(2.0, 2.6, 13), np.linspace(2.95, 4.0, 4)])
for L in [4, 8, 16, 32]:
    results = [simulate_checkerboard(T, L, n_cycles=10_000, n_warmup=1000) for T in tqdm(temperatures, desc=f"L = {L}")]
    np.savez(f"ising_checkerboard_L{L}.npz", temperature=temperatures, **{key: [r[key] for r in results] for key in results[0]})

In [ ]:
sizes = [4, 8, 16, 32]
datasets = {fr"${L} \times {L}$": np.load(f"ising_checkerboard_L{L}.npz") for L in sizes}
axes = plot_observables(datasets)
axes[0, 0].figure.suptitle("Finite-size effects", fontsize=15);

As $L$ increases, the energy and magnetization get closer to Onsager's solution, the drop of the
magnetization becomes sharper, and the peaks of the specific heat and of the susceptibility become
higher and narrower and move towards $T_c$. In the thermodynamic limit the specific heat has a
logarithmic divergence and the susceptibility diverges as $\chi \sim |T - T_c|^{-7/4}$.

A very useful quantity to locate the transition on finite systems is the **Binder cumulant**

\begin{equation}
  U_4 = 1 - \frac{\langle m^4 \rangle}{3 \langle m^2 \rangle^2}
\end{equation}

It goes to $2/3$ in the ordered phase (where $m$ is essentially $\pm m_0$) and to $0$ in the disordered phase
(where $m$ has a Gaussian distribution). At $T_c$, it takes a universal value, *independent of the system
size*. So the curves for different $L$ should all cross at $T_c$, and they approach a step function at
Onsager's $T_c$ as $L \to \infty$:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
for label, data in datasets.items():
    ax.plot(data["temperature"], data["binder"], "o-", ms=4.5, label=label)
ax.plot([1.0, T_C, T_C, 4.0], [2 / 3, 2 / 3, 0, 0], color="black", ls="--", lw=1.4, label=r"$L = \infty$")
ax.text(T_C - 0.16, 0.3, "$T_c$", color="black")
ax.set(xlabel="temperature $T/J$", ylabel="Binder cumulant $U_4$", title="Binder cumulant crossing")
ax.legend();

## Part 5: Autocorrelation time

Above we said that configurations that differ only by a spin flip are very correlated
and that a certain number of moves are needed to make two configurations
independent. As a rule of thumb, we used $L\times L$ updates to disentangle
configurations.

However, as we will see here, as one gets closer to the phase transition, it becomes
harder and harder to decorrelate two configurations. A way to measure this in
the disordered state is to compute the *autocorrelation time* through the quantity

\begin{equation}
  \mathcal{C}(t) = \langle m(t) \, m(0) \rangle - \langle m \rangle^2
\end{equation}

where $m(t)$ is the magnetization of the system at (Monte Carlo) time $t$.
If two configurations are completely decorrelated $\mathcal{C}(t)$ should
be zero. Usually $\mathcal{C}(t) \sim \exp(-t / \tau)$ where $\tau$ is the
autocorrelation time.

Write a code that computes the autocorrelation time at a given temperature
and study it as the temperature is reduced closer to the critical
temperature. You can start with an $8 \times 8$ lattice.

## Solution

We record the magnetization after each sweep, and compute $\mathcal{C}(t)$ as an average over all time
origins,

\begin{equation}
  \mathcal{C}(t) \simeq \frac{1}{n - t} \sum_{s=0}^{n-t-1} (m_{s+t} - \bar{m})(m_s - \bar{m})
\end{equation}

where $n$ is the number of measurements and $\bar m$ the average magnetization. This is equivalent to
$\langle m(t)\, m(0) \rangle - \langle m \rangle^2$, but more accurate numerically. We then fit
$\mathcal{C}(t)/\mathcal{C}(0)$ to $e^{-t/\tau}$ in the region where it is well above the noise.

In [ ]:
from scipy.optimize import curve_fit


def autocorrelation(x: np.ndarray, t_max: int) -> np.ndarray:
    """Normalized autocorrelation function C(t) / C(0) of the time series x, for t = 0, ..., t_max - 1."""
    x = x - x.mean()
    n = len(x)
    corr = np.array([np.dot(x[: n - t], x[t:]) / (n - t) for t in range(t_max)])
    return corr / corr[0]


def autocorrelation_time(corr: np.ndarray, threshold: float = 0.1) -> float:
    """Fit exp(-t / tau) to the autocorrelation function, as long as it stays above `threshold`."""
    below = np.flatnonzero(corr < threshold)
    n_fit = max(below[0] if len(below) else len(corr), 3)
    t = np.arange(n_fit)
    (tau,), _ = curve_fit(lambda t, tau: np.exp(-t / tau), t, corr[:n_fit], p0=(1.0,))
    return tau

In [ ]:
# production: this takes about 30 seconds
L, n_cycles, n_warmup, t_max = 8, 50_000, 1000, 250
temperatures_tau = np.array([2.3, 2.5, 2.75, 3.0, 3.5, 4.0])
correlations = {}

for T in tqdm(temperatures_tau, desc="temperatures"):
    config = Configuration(T=T, J=1.0, L=L, rng=rng)
    for _ in range(n_warmup):
        metropolis_sweep(config)
    mags = np.empty(n_cycles)
    for n in range(n_cycles):
        metropolis_sweep(config)
        mags[n] = config.magnetization / L**2
    correlations[T] = autocorrelation(mags, t_max)

taus = np.array([autocorrelation_time(correlations[T]) for T in temperatures_tau])
for T, tau in zip(temperatures_tau, taus):
    print(f"T = {T:.2f}:  tau = {tau:5.1f} sweeps")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), layout="constrained")
t = np.arange(t_max)
for (T, corr), tau, color in zip(correlations.items(), taus, PALETTE):
    ax1.semilogy(t, corr, "o", ms=3.5, color=color, label=f"$T = {T:g}$")
    ax1.semilogy(t, np.exp(-t / tau), "-", lw=1.2, color=color)
ax1.set(xlabel="Monte Carlo time $t$ (sweeps)", ylabel=r"$\mathcal{C}(t) / \mathcal{C}(0)$",
        ylim=(1e-2, 1.2), xlim=(0, 150), title=fr"Autocorrelation function, ${L} \times {L}$ lattice")
ax1.legend(fontsize=10, ncols=2, loc="lower right", frameon=True, facecolor="white", edgecolor="none")

ax2.plot(temperatures_tau, taus, "-", color="#898781", lw=1.2, zorder=1)
ax2.scatter(temperatures_tau, taus, s=45, c=PALETTE[: len(taus)], zorder=2)
ax2.axvline(T_C, color="#898781", lw=1)
ax2.text(T_C + 0.03, 0.03 * taus.max(), "$T_c$", color="#52514e")
ax2.set(xlim=(2.2, 4.1), xlabel="temperature $T/J$", ylabel=r"autocorrelation time $\tau$ (sweeps)",
        title="Critical slowing down");

The autocorrelation time increases sharply as the temperature is lowered towards $T_c$: this is
**critical slowing down**. In the thermodynamic limit, $\tau$ diverges at $T_c$ as $\tau \sim \xi^z$, where
$\xi$ is the correlation length and $z \simeq 2.17$ for the local Metropolis dynamics. On a finite lattice
$\xi$ is bounded by $L$, and $\tau \sim L^z$ at $T_c$: simulations of large systems close to the
transition become very expensive. Cluster algorithms are designed to overcome
this problem.